# Projekt 02 (medium): CNN-Bildklassifikation mit PyTorch — Architektur- und Regularisierungs-Ablation

**Ziel:** Von NumPy (Projekt 01) zu PyTorch. Du baust ein CNN für **Fashion-MNIST** (70 000 echte Produktbilder von Zalando, 10 Klassen, 28×28 Graustufen) und beantwortest experimentell drei Fragen aus dem Skript:

1. **Architektur:** Wie viel bringt der induktive Bias eines CNN (Lokalität + Weight Sharing, Skript 2.5) gegenüber einem MLP mit *mehr* Parametern?
2. **Regularisierung:** Was ändern Dropout, Weight Decay (AdamW) und Datenaugmentierung (Skript 2.3) an Lernkurven und Testfehler?
3. **Optimierer:** Adam vs. SGD+Momentum (Skript 2.1) bei identischem Budget.

Methodischer Anspruch: **Ablationsstudie** — es wird immer nur *eine* Stellgröße gegenüber einer Referenz verändert, sonst weiß man nicht, was den Unterschied verursacht hat.

> **Arbeitsweise:** Fülle die drei `TODO`-Stellen. Prüfzellen (Parameterzahl-Assert) und Erwartungswerte in den Texten sagen dir, ob du richtig liegst. Musterlösung mit allen ausgeführten Outputs: `loesung/loesung.ipynb`.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms

torch.manual_seed(42)
np.random.seed(42)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Rechne auf:", device)

DATA_DIR = "daten"   # torchvision laedt hierhin (per .gitignore vom Repo ausgeschlossen)

# Normalisierung mit den bekannten Fashion-MNIST-Statistiken (Mittel/Std der Trainingspixel)
normalize = transforms.Normalize((0.2860,), (0.3530,))
tf_plain = transforms.Compose([transforms.ToTensor(), normalize])
tf_augment = transforms.Compose([
    transforms.RandomCrop(28, padding=2),          # zufaellige Verschiebung um bis zu 2 Pixel
    transforms.RandomHorizontalFlip(),             # Spiegelung (fuer Kleidung eine gueltige Invarianz)
    transforms.ToTensor(), normalize,
])

train_plain = torchvision.datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=tf_plain)
train_aug   = torchvision.datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=tf_augment)
test_set    = torchvision.datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=tf_plain)

# Fuer die Ablation nutzen wir 20k der 60k Trainingsbilder — gleiche Aussagekraft, 1/3 Rechenzeit.
# (Fuer Bestleistung am Ende einfach SUBSET auf 60000 stellen.)
SUBSET = 20000
idx = torch.randperm(len(train_plain), generator=torch.Generator().manual_seed(0))[:SUBSET]

def make_loader(ds, train=True):
    if train:
        return DataLoader(Subset(ds, idx.tolist()), batch_size=128, shuffle=True)
    return DataLoader(ds, batch_size=512, shuffle=False)

loader_plain = make_loader(train_plain)
loader_aug = make_loader(train_aug)
loader_test = make_loader(test_set, train=False)

CLASSES = train_plain.classes
print(f"{SUBSET} Trainingsbilder, {len(test_set)} Testbilder, Klassen: {CLASSES}")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
raw = torchvision.datasets.FashionMNIST(DATA_DIR, train=True, download=True)  # ohne Transform, fuers Auge
for ax, i in zip(axes.ravel(), range(16)):
    img, label = raw[int(idx[i])]
    ax.imshow(img, cmap="gray"); ax.set_title(CLASSES[label], fontsize=8); ax.axis("off")
plt.suptitle("Fashion-MNIST: echte Zalando-Produktbilder (28x28 Graustufen)")
plt.tight_layout(); plt.show()

## Schritt 1: Die Kontrahenten — MLP vs. CNN

**MLP-Baseline:** $784 \to 256 \to 128 \to 10$, ReLU. Parameterzahl: $785{\cdot}256 + 257{\cdot}128 + 129{\cdot}10 \approx 235\,000$.

**CNN:** zwei Blöcke `[Conv3×3 → BatchNorm → ReLU] ×2 → MaxPool2×2` mit 32 bzw. 64 Kanälen, dann **Global Average Pooling** → Dropout → Linear(64→10). Nach Skript 2.5: Parameterzahl der Convs ist *unabhängig* von der Bildgröße; GAP erspart die fette Dense-Schicht. Rechne nach: das CNN hat nur $\approx 66\,000$ Parameter — **3,5× weniger als das MLP**.

Rezeptives Feld am Netzende (Skript 2.5, mit den zwei Pools als Stride-2-Stufen): jedes Neuron der letzten Conv-Schicht sieht ein $22{\times}22$-Fenster des Eingabebildes — fast das ganze Bild, wie es sein soll.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.MaxPool2d(2),
    )

class CNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        # TODO 1: Baue das CNN nach Spezifikation (Skript 2.5):
        #   features: conv_block(1, 32) -> conv_block(32, 64)     (28x28 -> 14x14 -> 7x7)
        #   head:     GlobalAveragePooling (nn.AdaptiveAvgPool2d(1)) -> nn.Flatten()
        #             -> nn.Dropout(dropout) -> nn.Linear(64, 10)
        self.features = ...
        self.head = ...

    def forward(self, x):
        return self.head(self.features(x))


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"MLP: {n_params(MLP()):>7,} Parameter")
print(f"CNN: {n_params(CNN()):>7,} Parameter")

# Selbstkontrolle: Parameterzahl nachrechnen (Skript 2.5: C_out * (C_in*k_h*k_w + 1) je Conv,
# 2*C je BatchNorm, 64*10+10 fuer den Kopf). Erwartet: 66_570.
assert n_params(CNN()) == 66_570, f"CNN hat {n_params(CNN()):,} Parameter, erwartet 66,570 — Architektur pruefen!"
print("Architektur-Check bestanden.")

## Schritt 2: Trainings- und Evaluationsschleife

Der Standard-PyTorch-Zyklus pro Batch: `zero_grad()` → Forward → Loss → `backward()` (das ist *exakt* dein Projekt-01-Code, nur automatisch differenziert) → `step()`.

Zwei Stolperfallen aus dem Skript, die hier real werden:
- `model.train()` / `model.eval()`: schaltet **Dropout** und **BatchNorm** um (Batch-Statistiken ↔ gleitende Mittel, Skript 2.4). Evaluation im Trainingsmodus liefert systematisch falsche Zahlen.
- `nn.CrossEntropyLoss` erwartet **rohe Logits** (Log-Sum-Exp-Trick intern, Skript 1.3) — kein Softmax im Modell!

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    # TODO 2: Eine Zeile fehlt hier. Ohne sie sind alle Testwerte systematisch falsch
    # (Skript 2.3/2.4 und Selbsttest-Frage 6). Welche — und warum genau?
    ...
    loss_sum, correct, n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss(reduction="sum")
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss_sum += criterion(logits, yb).item()
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += yb.numel()
    return loss_sum / n, correct / n

def train_model(model, opt, train_loader, epochs=3, log=True):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "test_loss": [], "test_acc": []}
    for epoch in range(1, epochs + 1):
        model.train()
        running, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            running += loss.item() * yb.numel(); n += yb.numel()
        te_loss, te_acc = evaluate(model, loader_test)
        hist["train_loss"].append(running / n)
        hist["test_loss"].append(te_loss)
        hist["test_acc"].append(te_acc)
        if log:
            print(f"  Epoche {epoch}: train_loss={running/n:.4f}  test_loss={te_loss:.4f}  test_acc={te_acc:.4f}")
    return hist


## Schritt 3: Die Ablation

Vier Konfigurationen, identisches Budget (gleiche Daten, gleiche Epochenzahl):

| # | Modell | Optimierer | Regularisierung | prüft Frage |
|---|---|---|---|---|
| A | MLP (235k Par.) | Adam ($10^{-3}$) | — | Architektur-Bias |
| B | CNN (66k Par.) | Adam ($10^{-3}$) | — | Architektur-Bias |
| C | CNN | **AdamW** ($10^{-3}$, wd $5{\cdot}10^{-4}$) | Dropout 0.3 + **Augmentierung** | Regularisierung |
| D | CNN | **SGD+Momentum** (0.1 / 0.9) | — | Optimierer |

In [ ]:
EPOCHS = 3
results = {}

# TODO 3: Vervollstaendige die vier Konfigurationen der Ablationstabelle oben.
# Tipp: torch.optim.Adam / AdamW (weight_decay=5e-4) / SGD (lr=0.1, momentum=0.9);
# Konfiguration C bekommt dropout=0.3 UND den Augmentierungs-Loader (loader_aug).
configs = {
    "A: MLP + Adam":            (lambda: MLP(),            lambda m: torch.optim.Adam(m.parameters(), lr=1e-3),  loader_plain),
    "B: CNN + Adam":            (lambda: CNN(dropout=0.0), lambda m: ...,  loader_plain),
    "C: CNN + AdamW + Reg/Aug": (lambda: ...,              lambda m: ...,  ...),
    "D: CNN + SGD-Momentum":    (lambda: CNN(dropout=0.0), lambda m: ...,  loader_plain),
}

models = {}
for name, (make_model, make_opt, loader) in configs.items():
    torch.manual_seed(7)                      # gleiche Initialisierung, wo die Architektur gleich ist
    print(name)
    t0 = time.time()
    model = make_model()
    results[name] = train_model(model, make_opt(model), loader, epochs=EPOCHS)
    models[name] = model
    print(f"  ({time.time() - t0:.0f} s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in results.items():
    ep = range(1, EPOCHS + 1)
    axes[0].plot(ep, h["test_loss"], marker="o", label=name)
    axes[1].plot(ep, h["test_acc"], marker="o", label=name)
axes[0].set_xlabel("Epoche"); axes[0].set_ylabel("Test-Loss"); axes[0].set_title("Test-Loss"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("Epoche"); axes[1].set_ylabel("Test-Accuracy"); axes[1].set_title("Test-Accuracy"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'Konfiguration':<28} {'Parameter':>10} {'Test-Acc':>9}")
for name, h in results.items():
    print(f"{name:<28} {n_params(models[name]):>10,} {h['test_acc'][-1]:>9.4f}")

**Interpretation (vergleiche mit deinen Zahlen):**

- **A vs. B:** Das CNN schlägt das MLP trotz ~3,5× weniger Parametern — der induktive Bias (Translationsäquivarianz, Lokalität) ist bei Bildern mehr wert als rohe Kapazität. Parameterzahl ist *kein* Maß für Ausdrucksstärke auf strukturierten Daten.
- **B vs. C:** Nach nur 3 Epochen liegt die regularisierte Variante oft noch *nicht* vorn — Augmentierung macht das Trainingsproblem schwerer und zahlt sich erst über mehr Epochen aus (Regularisierung tauscht Trainingsfit gegen Generalisierung). Miss den Effekt in Schritt 5 über längeres Training.
- **B vs. D:** SGD+Momentum kann mit gut gewählter Lernrate mit Adam mithalten (bei Bildklassifikation historisch sogar oft besser bei langem Training); Adam ist robuster gegenüber der Lernratenwahl — genau der Trade-off aus Skript 2.1.

## Schritt 4: Fehlerdiagnose — wo irrt das Netz?

Accuracy allein versteckt die Struktur der Fehler. Confusion Matrix + die am sichersten falsch klassifizierten Bilder zeigen, *welche* Klassen das Netz verwechselt (klassisch: Shirt ↔ T-Shirt ↔ Pullover — die sind auch für Menschen schwer).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_name = max(results, key=lambda k: results[k]["test_acc"][-1])
best = models[best_name]
print("Bestes Modell:", best_name)

best.eval()
all_logits, all_y = [], []
with torch.no_grad():
    for xb, yb in loader_test:
        all_logits.append(best(xb.to(device)).cpu())
        all_y.append(yb)
logits = torch.cat(all_logits); y_true = torch.cat(all_y)
y_pred = logits.argmax(dim=1)

fig, ax = plt.subplots(figsize=(7.5, 7))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout(); plt.show()

# Die "selbstsichersten" Fehler: falsch UND mit hoher Konfidenz
probs = logits.softmax(dim=1)
conf, _ = probs.max(dim=1)
wrong = (y_pred != y_true).nonzero().squeeze(1)
worst = wrong[conf[wrong].argsort(descending=True)][:8]
fig, axes = plt.subplots(1, 8, figsize=(14, 2.4))
for ax, i in zip(axes, worst.tolist()):
    img, _ = test_set[i]
    ax.imshow(img.squeeze() * 0.3530 + 0.2860, cmap="gray")
    ax.set_title(f"wahr: {CLASSES[y_true[i]]}\npred: {CLASSES[y_pred[i]]} ({conf[i]:.2f})", fontsize=7)
    ax.axis("off")
plt.suptitle("Konfidenteste Fehlklassifikationen"); plt.tight_layout(); plt.show()

## Schritt 5: Zahlt sich Regularisierung über die Zeit aus?

Die 3-Epochen-Ablation ist eine Momentaufnahme. Jetzt der eigentliche Test aus Skript 2.3: **CNN pur vs. CNN + Reg/Aug über 10 Epochen.** Erwartung: Das unregularisierte Netz fittet den Train-Loss schneller Richtung 0, aber sein Test-Loss stagniert oder steigt (Overfitting-Schere); die regularisierte Variante lernt langsamer und endet besser.

In [ ]:
EPOCHS_LONG = 10
long_results = {}
for name, (make_model, make_opt, loader) in {
    "CNN pur":       configs["B: CNN + Adam"],
    "CNN + Reg/Aug": configs["C: CNN + AdamW + Reg/Aug"],
}.items():
    torch.manual_seed(7)
    print(name)
    model = make_model()
    long_results[name] = train_model(model, make_opt(model), loader, epochs=EPOCHS_LONG, log=False)
    print(f"  finale Test-Acc: {long_results[name]['test_acc'][-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in long_results.items():
    ep = range(1, EPOCHS_LONG + 1)
    axes[0].plot(ep, h["train_loss"], "--", label=f"{name} (Train)")
    axes[0].plot(ep, h["test_loss"], label=f"{name} (Test)")
    axes[1].plot(ep, h["test_acc"], marker="o", label=name)
axes[0].set_xlabel("Epoche"); axes[0].set_ylabel("Loss"); axes[0].set_title("Overfitting-Schere: Train- vs. Test-Loss"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("Epoche"); axes[1].set_ylabel("Test-Accuracy"); axes[1].set_title("Test-Accuracy"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Fazit

- **Induktiver Bias schlägt Parameterzahl:** Das 66k-CNN dominiert das 235k-MLP — Architektur kodiert Vorwissen über die Datenstruktur.
- **Regularisierung ist eine Investition:** kurzfristig langsamer, langfristig kleinere Train/Test-Schere. Die Wirkung *muss* man über Lernkurven beurteilen, nicht über eine einzelne Zahl.
- **`model.eval()` ist keine Formalie:** BatchNorm und Dropout verhalten sich in Training und Inferenz fundamental verschieden (Skript 2.3/2.4).
- **Ablation als Methode:** immer nur eine Stellgröße ändern — das trägt durch alle folgenden Module (und jede echte Forschungsarbeit).

**Weiter in Projekt 03:** unüberwachtes Lernen auf echten Kundendaten — dort gibt es keine Labels und damit keine Accuracy mehr; Validierung wird zur eigentlichen Herausforderung.